# NB106 — The Analytic CP

## Goal

Derive a closed-form expression for the CP ratio from {2,3,5,7} alone — no integration.

## Key insight from NB105

1. Each CRT sector has ONE crossing per primorial window of 210
2. ALL CP asymmetry lives in window 0; windows w≥1 contribute CP = 1.000
3. At window 0, R_k(ci; br) = R_ss(ci; lower ICs) + 2π·j_{k+1}·e^{-κ(ci+1)}
4. g1 crossings (ci=11, 31) are inside the wrapping horizon → wrapping modifies R²
5. g2 crossings (ci=61, 191) are outside → no wrapping, R_wrapped = R_raw

## Strategy

The sector RMS² = (1/N) Σ_br R_wrapped(ci; br)² summed over all 210 branches.

For **g2** (no wrapping): RMS² = (1/N) Σ_br [R_ss + 2πj·e^{-κ(ci+1)}]²
For **g1** (with wrapping): RMS² = (1/N) Σ_br [wrap(R_ss + 2πj·e^{-κ(ci+1)})]²

where wrap(x) = ((x + π) mod 2π) − π.

If R_ss is approximately constant across branches at a given crossing, the sum
decomposes into a tractable form indexed by j = 0, 1, ..., p_{k+1}−1.

In [1]:
# ── S0: Setup ──
import sys, time, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from math import gcd

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               X4, X3, X2, LAM7, X4_LEP,
                               CP_PAIRS, SM_TARGETS, PHYSICAL_CROSSINGS, DLOG)
from solenoid_system import SolenoidSystem

PRIMES = SA.primes
P = SA.P
PHI = SA.PHI

ss = SolenoidSystem(PRIMES, OMEGA, EPSILON, KAPPA)
branches = SA.all_branches()
T_MAX = 5001

coprime_cis = SA.coprime_indices(T_MAX)
ci_a3, ci_a5, ci_a7 = SA.sector_labels(coprime_cis)
t_eval = coprime_cis.astype(float) + 1.0

# Integrate
t0 = time.time()
results = ss.integrate_all_branches(branches, t_eval, T_MAX, backend='jax')
dt = time.time() - t0
print(f"Integration: {dt:.1f}s")

# Stack and wrap
R_stack = np.stack([results[b] for b in branches])  # (210, n_ci, 4)
R_wrapped = np.mod(R_stack, 2 * np.pi)
R_wrapped[R_wrapped > np.pi] -= 2 * np.pi

branches_arr = np.array(branches)
n_br = len(branches)

# Physical crossing positions (from NB105)
CROSS = {'QUARK_g1': 11, 'QUARK_g2': 191, 'LEPTON_g1': 31, 'LEPTON_g2': 61}
print(f"Physical crossings: {CROSS}")
print(f"kappa = {KAPPA:.6f}, exp(-kappa*12) = {np.exp(-KAPPA*12):.6f}")

  JAX [CPU (1 device(s))]: 210 branches, 1143 eval pts, T=5001 — 26.05s
Integration: 28.2s
Physical crossings: {'QUARK_g1': 11, 'QUARK_g2': 191, 'LEPTON_g1': 31, 'LEPTON_g2': 61}
kappa = 0.069007, exp(-kappa*12) = 0.436888


## S1: R Decomposition at Physical Crossings

At crossing ci, branch br = (j₁, j₂, j₃, j₄):

R_k(ci; br) = R_k^ss(ci; j₁,...,j_k) + 2π · j_{k+1} · exp(−κ·(ci+1))

The transient depends only on j_{k+1}, while R_ss depends on the lower ICs j₁,...,j_k.

For level 3 (the mass-critical level): j_{k+1} = j₄, lower ICs = (j₁,j₂,j₃).
There are 30 distinct (j₁,j₂,j₃) combinations and 7 j₄ values (0–6).

**Key question**: Is R_ss the same for all (j₁,j₂,j₃) at the physical crossing? Or does it vary?

In [2]:
# ── S1: R_ss distribution at physical crossings (level 3) ──
# For each physical crossing, extract R_3 for all 210 branches.
# Group by j4 to separate transient from steady-state.

lev = 3

# Find the coprime crossing index for each physical crossing
def find_ci_idx(ci_target):
    """Find index into coprime_cis array for a given ci value."""
    idx = np.where(coprime_cis == ci_target)[0]
    return idx[0] if len(idx) > 0 else None

for name, ci in CROSS.items():
    ci_idx = find_ci_idx(ci)
    if ci_idx is None:
        print(f"{name}: ci={ci} NOT a coprime crossing!")
        continue
    
    R_raw = R_stack[:, ci_idx, lev]  # (210,) raw R_3 values
    
    # Group by j4 to extract R_ss (the base at j4=0)
    print(f"\n{name} (ci={ci}, t={ci+1}):")
    print(f"  Decay factor: exp(-kappa*(ci+1)) = {np.exp(-KAPPA*(ci+1)):.6f}")
    print(f"  Max transient: 2*pi*6*decay = {2*np.pi*6*np.exp(-KAPPA*(ci+1)):.4f}")
    print(f"  {'j4':>3} | {'R_raw mean':>12} {'R_raw std':>12} | {'R_ss (j4=0)':>12} {'transient':>10}")
    print(f"  {'-'*60}")
    
    # R_ss for each (j1,j2,j3) = R at j4=0
    j4_0_mask = branches_arr[:, 3] == 0  # 30 branches with j4=0
    R_ss_vals = R_raw[j4_0_mask]
    
    for j4 in range(7):
        j4_mask = branches_arr[:, 3] == j4
        R_j4 = R_raw[j4_mask]  # 30 values for this j4
        
        # Expected: R = R_ss + 2*pi*j4*exp(-kappa*(ci+1))
        transient = 2 * np.pi * j4 * np.exp(-KAPPA * (ci + 1))
        expected = R_ss_vals + transient  # should match R_j4
        
        match_err = np.max(np.abs(R_j4 - expected))
        
        print(f"  {j4:3d} | {np.mean(R_j4):12.6f} {np.std(R_j4):12.6f} | "
              f"{np.mean(R_ss_vals):12.6f} {transient:10.4f} "
              f"(err={match_err:.2e})")
    
    # Show the R_ss distribution for this crossing
    print(f"\n  R_ss statistics: mean={np.mean(R_ss_vals):.6f}, std={np.std(R_ss_vals):.6f}")
    print(f"  R_ss range: [{np.min(R_ss_vals):.6f}, {np.max(R_ss_vals):.6f}]")
    print(f"  |R_ss| < pi for all: {np.all(np.abs(R_ss_vals) < np.pi)}")


QUARK_g1 (ci=11, t=12):
  Decay factor: exp(-kappa*(ci+1)) = 0.436888
  Max transient: 2*pi*6*decay = 16.4703
   j4 |   R_raw mean    R_raw std |  R_ss (j4=0)  transient
  ------------------------------------------------------------
    0 |     0.859821     0.325553 |     0.859821     0.0000 (err=0.00e+00)
    1 |     3.604868     0.325553 |     0.859821     2.7450 (err=5.33e-15)
    2 |     6.349916     0.325553 |     0.859821     5.4901 (err=8.88e-15)
    3 |     9.094963     0.325553 |     0.859821     8.2351 (err=1.24e-14)
    4 |    11.840011     0.325553 |     0.859821    10.9802 (err=1.78e-14)
    5 |    14.585058     0.325553 |     0.859821    13.7252 (err=1.78e-14)
    6 |    17.330106     0.325553 |     0.859821    16.4703 (err=2.84e-14)

  R_ss statistics: mean=0.859821, std=0.325553
  R_ss range: [0.416544, 1.597157]
  |R_ss| < pi for all: True

QUARK_g2 (ci=191, t=192):
  Decay factor: exp(-kappa*(ci+1)) = 0.000002
  Max transient: 2*pi*6*decay = 0.0001
   j4 |   R_raw me

## S2: Window-0 Anatomy — Why All Asymmetry Lives Here

NB105 proved: per-window CP = 1.000 for w ≥ 1 at all 4 levels. So the entire CP asymmetry
comes from window 0 (crossings 1–209). We now decompose the window-0 sum-of-squares for
each sector to understand what drives the CP ratio.

For a sector (a₃, a₅=0, a₇), the sector RMS² sums R²_wrapped over all coprime crossings
matching that sector AND all 210 branches. The g1 and g2 sub-sectors of a CP pair share
the same a₃ but differ in a₇. 

**The question**: What fraction of the sector RMS² comes from window 0, and can we express
the window-0 contribution analytically?

In [4]:
# ── S2: Window-0 sector R² decomposition at level 3 ──
# Manual accumulation matching accumulate_sectors logic, but from stacked array.

lev = 3
N_br, N_ci, N_lev = R_stack.shape  # (210, 1143, 4)

def sector_cp_from_stack(R_stk, cis, a3_arr, a5_arr, a7_arr, n_br):
    """Compute sector RMS and CP ratios from stacked R array."""
    R_w = np.mod(R_stk, 2*np.pi)
    R_w[R_w > np.pi] -= 2*np.pi
    
    R_sq_sum = (R_w**2).sum(axis=0)   # (n_ci, n_lev)
    
    sector_sq = np.zeros((4, 2, 6, 4))
    sector_cnt = np.zeros((4, 2, 6), dtype=int)
    
    for idx in range(len(cis)):
        a5, a3, a7 = int(a5_arr[idx]), int(a3_arr[idx]), int(a7_arr[idx])
        sector_sq[a5, a3, a7] += R_sq_sum[idx]
        sector_cnt[a5, a3, a7] += n_br
    
    sector_rms = np.zeros_like(sector_sq)
    for a5 in range(4):
        for a3 in range(2):
            for a7 in range(6):
                cnt = sector_cnt[a5, a3, a7]
                if cnt > 0:
                    sector_rms[a5, a3, a7] = np.sqrt(sector_sq[a5, a3, a7] / cnt)
    
    cp = {}
    for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
        r1 = sector_rms[0, a3, a7g1, lev]
        r2 = sector_rms[0, a3, a7g2, lev]
        cp[ch] = r1 / r2 if r2 > 0 else np.inf
    return sector_rms, cp

# --- Full accumulation ---
full_rms, cp_full = sector_cp_from_stack(
    R_stack, coprime_cis, ci_a3, ci_a5, ci_a7, N_br)
print("Full CP ratios (all windows):")
for ch, v in cp_full.items():
    print(f"  {ch}: {v:.6f}")

# --- Window 0 only (ci < 210) ---
w0_mask = coprime_cis < 210
w0_cis = coprime_cis[w0_mask]
w0_R = R_stack[:, w0_mask, :]

w0_rms, cp_w0 = sector_cp_from_stack(
    w0_R, w0_cis, ci_a3[w0_mask], ci_a5[w0_mask], ci_a7[w0_mask], N_br)
print("\nWindow-0 CP ratios:")
for ch, v in cp_w0.items():
    print(f"  {ch}: {v:.6f}")

# --- Window >= 1 only ---
wN_mask = coprime_cis >= 210
wN_cis = coprime_cis[wN_mask]
wN_R = R_stack[:, wN_mask, :]

wN_rms, cp_wN = sector_cp_from_stack(
    wN_R, wN_cis, ci_a3[wN_mask], ci_a5[wN_mask], ci_a7[wN_mask], N_br)
print("\nWindow >= 1 CP ratios:")
for ch, v in cp_wN.items():
    print(f"  {ch}: {v:.6f}")

# --- Crossings per sector in window 0 ---
print("\n--- Coprime crossings per CRT sector (a5=0) in window 0 ---")
for a3 in [0, 1]:
    for a7 in range(6):
        mask_s = ((ci_a3[w0_mask] == a3)
                  & (ci_a5[w0_mask] == 0)
                  & (ci_a7[w0_mask] == a7))
        cis_in = w0_cis[mask_s]
        if len(cis_in) > 0:
            label = ""
            for name, ci_v in CROSS.items():
                if ci_v in cis_in:
                    label += f"  <-- {name}"
            print(f"  (a3={a3}, a7={a7}): {len(cis_in)} crossings: "
                  f"{list(cis_in[:8])}{label}")

# --- Sector R^2 fraction from window 0 ---
print("\n--- Sector R^2 fraction from window 0 (a5=0, level 3) ---")
for a3 in [0, 1]:
    for a7 in range(6):
        f_sq = full_rms[0, a3, a7, lev]**2
        w0_sq = w0_rms[0, a3, a7, lev]**2
        if f_sq > 0:
            frac = w0_sq / f_sq
            print(f"  (a3={a3}, a7={a7}): w0={w0_sq:.6f}  "
                  f"full={f_sq:.6f}  frac={frac:.4f}")

Full CP ratios (all windows):
  QUARK: 1.538827
  LEPTON: 1.942172

Window-0 CP ratios:
  QUARK: 5.812435
  LEPTON: 6.177116

Window >= 1 CP ratios:
  QUARK: 1.000005
  LEPTON: 0.999998

--- Coprime crossings per CRT sector (a5=0) in window 0 ---
  (a3=0, a7=0): 1 crossings: [np.int64(1)]
  (a3=0, a7=1): 1 crossings: [np.int64(31)]  <-- LEPTON_g1
  (a3=0, a7=2): 1 crossings: [np.int64(121)]
  (a3=0, a7=3): 1 crossings: [np.int64(181)]
  (a3=0, a7=4): 1 crossings: [np.int64(151)]
  (a3=0, a7=5): 1 crossings: [np.int64(61)]  <-- LEPTON_g2
  (a3=1, a7=0): 1 crossings: [np.int64(71)]
  (a3=1, a7=1): 1 crossings: [np.int64(101)]
  (a3=1, a7=2): 1 crossings: [np.int64(191)]  <-- QUARK_g2
  (a3=1, a7=3): 1 crossings: [np.int64(41)]
  (a3=1, a7=4): 1 crossings: [np.int64(11)]  <-- QUARK_g1
  (a3=1, a7=5): 1 crossings: [np.int64(131)]

--- Sector R^2 fraction from window 0 (a5=0, level 3) ---
  (a3=0, a7=0): w0=3.528233  full=0.202384  frac=17.4333
  (a3=0, a7=1): w0=4.088254  full=0.225720  fr

## S3: The Dilution Formula from First Principles

The sector RMS² sums R²_wrapped over all branches and all crossings in the sector.
Since each (a₃, a₅=0, a₇) sector has exactly **one crossing per 210-window**, we
can decompose:

$$S_{g1} = S_{g1}^{w0} + S_{g1}^{w \ge 1}$$

Since CP = 1 for w ≥ 1:  $S_{g1}^{w \ge 1} \approx S_{g2}^{w \ge 1} = S^{w \ge 1}$

$$\text{CP}^2 = \frac{S_{g1}^{w0} + S^{w \ge 1}}{S_{g2}^{w0} + S^{w \ge 1}}
= \frac{C_0^2 + r}{1 + r}$$

where $C_0^2 = S_{g1}^{w0} / S_{g2}^{w0}$ and $r = S^{w \ge 1} / S_{g2}^{w0}$.

This **derives** the NB97 dilution formula from the window structure. Now we need to
express $C_0$ and $r$ as functions of {2, 3, 5, 7} alone.

In [5]:
# ── S3a: Extract C₀ and r from window decomposition ──
# Verify the dilution formula: CP² = (C₀² + r) / (1 + r)

for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
    # Window-0 sums (1 crossing per sector, sum over 210 branches)
    # R² sum at the single w0 crossing for each sector
    ci_g1 = CROSS[f"{ch}_g1"]
    ci_g2 = CROSS[f"{ch}_g2"]
    
    ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
    ci_g2_idx = np.where(coprime_cis == ci_g2)[0][0]
    
    # Wrapped R at level 3 for all branches
    R_g1 = R_wrapped[:, ci_g1_idx, lev]  # (210,) already wrapped
    R_g2 = R_wrapped[:, ci_g2_idx, lev]
    
    S_g1_w0 = np.sum(R_g1**2)
    S_g2_w0 = np.sum(R_g2**2)
    C0_sq = S_g1_w0 / S_g2_w0
    C0 = np.sqrt(C0_sq)
    
    # Window >= 1 sum: all crossings with matching (a3, a5=0, a7) in w >= 1
    # For g2 sector:
    wN_g2_mask = ((coprime_cis >= 210) & 
                  (ci_a3 == a3) & (ci_a5 == 0) & (ci_a7 == a7g2))
    n_windows_g2 = np.sum(wN_g2_mask)
    
    # Sum R²_wrapped over all branches and all w>=1 crossings in g2 sector
    R_wN_g2 = R_wrapped[:, wN_g2_mask, lev]  # (210, n_windows_g2)
    S_wN_g2 = np.sum(R_wN_g2**2)
    
    # Same for g1 sector
    wN_g1_mask = ((coprime_cis >= 210) & 
                  (ci_a3 == a3) & (ci_a5 == 0) & (ci_a7 == a7g1))
    n_windows_g1 = np.sum(wN_g1_mask)
    R_wN_g1 = R_wrapped[:, wN_g1_mask, lev]
    S_wN_g1 = np.sum(R_wN_g1**2)
    
    # Verify: S_wN_g1 ≈ S_wN_g2 (since CP=1 for w>=1)
    S_wN = (S_wN_g1 + S_wN_g2) / 2  # average
    asymmetry = abs(S_wN_g1 - S_wN_g2) / S_wN
    
    r = S_wN / S_g2_w0
    
    # Dilution formula prediction
    CP_pred = np.sqrt((C0_sq + r) / (1 + r))
    
    print(f"\n{ch}:")
    print(f"  ci_g1={ci_g1}, ci_g2={ci_g2}")
    print(f"  S_g1_w0 = {S_g1_w0:.4f}")
    print(f"  S_g2_w0 = {S_g2_w0:.4f}")
    print(f"  C₀ = {C0:.6f}  (C₀² = {C0_sq:.4f})")
    print(f"  S_wN_g1 = {S_wN_g1:.4f}  ({n_windows_g1} windows)")
    print(f"  S_wN_g2 = {S_wN_g2:.4f}  ({n_windows_g2} windows)")
    print(f"  g1/g2 wN asymmetry = {asymmetry:.6f}")
    print(f"  r = S_wN / S_g2_w0 = {r:.6f}")
    print(f"  Dilution formula: CP = sqrt(({C0_sq:.4f} + {r:.4f}) / "
          f"(1 + {r:.4f})) = {CP_pred:.6f}")
    print(f"  Actual CP (full)  = {cp_full[ch]:.6f}")
    print(f"  Match: {abs(CP_pred - cp_full[ch]) / cp_full[ch] * 100:.4f}%")


QUARK:
  ci_g1=11, ci_g2=191
  S_g1_w0 = 688.1937
  S_g2_w0 = 20.3702
  C₀ = 5.812435  (C₀² = 33.7844)
  S_wN_g1 = 467.8155  (23 windows)
  S_wN_g2 = 447.4709  (22 windows)
  g1/g2 wN asymmetry = 0.044455
  r = S_wN / S_g2_w0 = 22.466351
  Dilution formula: CP = sqrt((33.7844 + 22.4664) / (1 + 22.4664)) = 1.548251
  Actual CP (full)  = 1.538827
  Match: 0.6124%

LEPTON:
  ci_g1=31, ci_g2=61
  S_g1_w0 = 858.5334
  S_g2_w0 = 22.5002
  C₀ = 6.177116  (C₀² = 38.1568)
  S_wN_g1 = 279.0944  (23 windows)
  S_wN_g2 = 279.0954  (23 windows)
  g1/g2 wN asymmetry = 0.000003
  r = S_wN / S_g2_w0 = 12.404129
  Dilution formula: CP = sqrt((38.1568 + 12.4041) / (1 + 12.4041)) = 1.942174
  Actual CP (full)  = 1.942172
  Match: 0.0001%


In [6]:
# ── S3b: Wrapped sum anatomy per  j4 slice ──
# For each physical crossing, decompose the wrapped R² sum by j4.
# Are the contributions uniform across j4, or is there structure?

for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
    ci_g1 = CROSS[f"{ch}_g1"]
    ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
    
    alpha_g1 = np.exp(-KAPPA * (ci_g1 + 1))
    Delta = 2 * np.pi * alpha_g1  # step per j4
    
    print(f"\n{'='*65}")
    print(f"{ch} g1 (ci={ci_g1}): alpha={alpha_g1:.6f}, Delta={Delta:.4f}")
    print(f"  7*Delta = {7*Delta:.4f}, 7*Delta/(2pi) = {7*Delta/(2*np.pi):.4f} wraps")
    print(f"  {'j4':>3} {'mean(R)':>10} {'mean(wrap(R))':>14} {'sum(wrap²)':>12} {'frac':>8}")
    
    total_sq = 0
    per_j4_sq = []
    for j4 in range(7):
        j4_mask = branches_arr[:, 3] == j4  # 30 branches
        R_raw = R_stack[j4_mask, ci_g1_idx, lev]  # (30,)
        R_wr = np.mod(R_raw, 2*np.pi)
        R_wr[R_wr > np.pi] -= 2*np.pi
        
        sq = np.sum(R_wr**2)
        per_j4_sq.append(sq)
        total_sq += sq
        
        print(f"  {j4:3d} {np.mean(R_raw):10.4f} {np.mean(R_wr):14.4f} "
              f"{sq:12.4f} {sq/688 if ch=='QUARK' else sq/859:8.4f}")
    
    print(f"\n  Total sum(wrap²) = {total_sq:.4f}")
    print(f"  Per-j4 std/mean = {np.std(per_j4_sq)/np.mean(per_j4_sq):.4f}")
    
    # Is the wrapped sum sensitive to R_ss?
    # Check: for j4=0 (no transient), all R² comes from R_ss
    j4_0_mask = branches_arr[:, 3] == 0
    R_ss_vals = R_stack[j4_0_mask, ci_g1_idx, lev]  # R_ss for 30 branches
    print(f"\n  R_ss (j4=0): mean={np.mean(R_ss_vals):.4f}, "
          f"std={np.std(R_ss_vals):.4f}")
    print(f"  sum(R_ss²) = {np.sum(R_ss_vals**2):.4f}")

# --- Also check g2 for comparison ---
print(f"\n{'='*65}")
print("g2 crossings (no wrapping, steady-state dominated):")
for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
    ci_g2 = CROSS[f"{ch}_g2"]
    ci_g2_idx = np.where(coprime_cis == ci_g2)[0][0]
    R_g2 = R_wrapped[:, ci_g2_idx, lev]
    print(f"  {ch} g2 (ci={ci_g2}): sum(R²)={np.sum(R_g2**2):.4f}, "
          f"mean(R)={np.mean(R_g2):.6f}, std(R)={np.std(R_g2):.6f}")


QUARK g1 (ci=11): alpha=0.436888, Delta=2.7450
  7*Delta = 19.2153, 7*Delta/(2pi) = 3.0582 wraps
   j4    mean(R)  mean(wrap(R))   sum(wrap²)     frac
    0     0.8598         0.8598      25.3583   0.0369
    1     3.6049        -2.6783     218.3810   0.3174
    2     6.3499         0.0667       3.3131   0.0048
    3     9.0950         1.5551     224.2599   0.3260
    4    11.8400        -0.7264      19.0075   0.0276
    5    14.5851         2.0187     125.4325   0.1823
    6    17.3301        -1.5195      72.4414   0.1053

  Total sum(wrap²) = 688.1937
  Per-j4 std/mean = 0.8794

  R_ss (j4=0): mean=0.8598, std=0.3256
  sum(R_ss²) = 25.3583

LEPTON g1 (ci=31): alpha=0.109897, Delta=0.6905
  7*Delta = 4.8335, 7*Delta/(2pi) = 0.7693 wraps
   j4    mean(R)  mean(wrap(R))   sum(wrap²)     frac
    0     0.5716         0.5716      16.9944   0.0198
    1     1.2621         1.2621      54.9796   0.0640
    2     1.9526         1.9526     121.5725   0.1415
    3     2.6431         1.3865    

In [7]:
# ── S3c: Does the wrapped sum depend on R_ss? ──
# Compare: (a) actual wrapped sum across 30 branch offsets
#          (b) hypothetical R_ss=0 (pure lattice sum × 30)
#          (c) hypothetical R_ss=mean(R_ss) for all branches

def wrapped_sq_sum(R_ss_arr, Delta, n_j4=7):
    """Σ_{j4=0}^{n_j4-1} Σ_br wrap(R_ss[br] + j4*Delta)²"""
    total = 0.0
    for j4 in range(n_j4):
        R = R_ss_arr + j4 * Delta
        W = np.mod(R, 2*np.pi)
        W[W > np.pi] -= 2*np.pi
        total += np.sum(W**2)
    return total

def pure_lattice_sum(Delta, n_j4=7):
    """Σ_{j4=0}^{n_j4-1} wrap(j4*Delta)²  (single point, R_ss=0)"""
    j4s = np.arange(n_j4)
    V = j4s * Delta
    W = np.mod(V, 2*np.pi)
    W[W > np.pi] -= 2*np.pi
    return np.sum(W**2)

print("="*70)
print("SENSITIVITY OF C₀ TO R_ss DISTRIBUTION")
print("="*70)

for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
    ci_g1 = CROSS[f"{ch}_g1"]
    ci_g2 = CROSS[f"{ch}_g2"]
    
    alpha_g1 = np.exp(-KAPPA * (ci_g1 + 1))
    Delta_g1 = 2 * np.pi * alpha_g1
    
    # Get actual R_ss (j4=0 branches) at g1 crossing
    j4_0 = branches_arr[:, 3] == 0
    ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
    ci_g2_idx = np.where(coprime_cis == ci_g2)[0][0]
    R_ss_g1 = R_stack[j4_0, ci_g1_idx, lev]  # 30 values
    
    # (a) Actual: use true R_ss distribution
    S_actual = wrapped_sq_sum(R_ss_g1, Delta_g1)
    
    # (b) R_ss = 0 for all branches → pure lattice × 30
    S_zero = 30 * pure_lattice_sum(Delta_g1)
    
    # (c) R_ss = mean for all branches
    R_ss_mean = np.full_like(R_ss_g1, np.mean(R_ss_g1))
    S_mean = wrapped_sq_sum(R_ss_mean, Delta_g1)
    
    # (d) R_ss uniform random in [-pi, pi] → expected value
    np.random.seed(42)
    S_random_samples = []
    for _ in range(1000):
        R_rand = np.random.uniform(-np.pi, np.pi, 30)
        S_random_samples.append(wrapped_sq_sum(R_rand, Delta_g1))
    S_random = np.mean(S_random_samples)
    
    # g2 sum for C₀ computation
    R_g2 = R_wrapped[:, ci_g2_idx, lev]
    S_g2 = np.sum(R_g2**2)
    
    print(f"\n{ch} (ci_g1={ci_g1}, Delta={Delta_g1:.4f}):")
    print(f"  S_g2_w0 = {S_g2:.2f}")
    print(f"  (a) Actual R_ss:     S = {S_actual:.2f} → C₀ = {np.sqrt(S_actual/S_g2):.4f}")
    print(f"  (b) R_ss = 0:        S = {S_zero:.2f} → C₀ = {np.sqrt(S_zero/S_g2):.4f}")
    print(f"  (c) R_ss = mean:     S = {S_mean:.2f} → C₀ = {np.sqrt(S_mean/S_g2):.4f}")
    print(f"  (d) R_ss ~ U[-π,π]:  S = {S_random:.2f} → C₀ = {np.sqrt(S_random/S_g2):.4f}")
    print(f"  Pure lattice sum (7 points): {pure_lattice_sum(Delta_g1):.4f}")
    print(f"  Expected uniform sum (per br): {S_random/30/7:.4f} vs π²/3 = {np.pi**2/3:.4f}")

SENSITIVITY OF C₀ TO R_ss DISTRIBUTION

QUARK (ci_g1=11, Delta=2.7450):
  S_g2_w0 = 20.37
  (a) Actual R_ss:     S = 688.19 → C₀ = 5.8124
  (b) R_ss = 0:        S = 644.83 → C₀ = 5.6263
  (c) R_ss = mean:     S = 682.04 → C₀ = 5.7864
  (d) R_ss ~ U[-π,π]:  S = 690.77 → C₀ = 5.8233
  Pure lattice sum (7 points): 21.4943
  Expected uniform sum (per br): 3.2894 vs π²/3 = 3.2899

LEPTON (ci_g1=31, Delta=0.6905):
  S_g2_w0 = 22.50
  (a) Actual R_ss:     S = 858.53 → C₀ = 6.1771
  (b) R_ss = 0:        S = 806.90 → C₀ = 5.9885
  (c) R_ss = mean:     S = 869.46 → C₀ = 6.2163
  (d) R_ss ~ U[-π,π]:  S = 691.39 → C₀ = 5.5433
  Pure lattice sum (7 points): 26.8968
  Expected uniform sum (per br): 3.2923 vs π²/3 = 3.2899


## S4: The Lattice Sum Formula

For `p₄ = 7` copies of j₄, the window-0 g1 sum decomposes into a **lattice sum**.
Each branch has R = R_ss + j₄ · Δ where Δ = 2π · exp(−κ(ci+1)). 
Wrapping R to [−π, π] produces:

$$L(\Delta, p) = \sum_{j=0}^{p-1} \mathrm{wrap}(j\Delta)^2 
= S_2 \Delta^2 - 4\pi\Delta M_1 + 4\pi^2 M_0$$

where $S_2 = p(p-1)(2p-1)/6$, and $M_0 = \sum n_j^2$, $M_1 = \sum j \cdot n_j$
are **integer-valued coefficients** from the wrapping count pattern $n_j = \text{round}(j\alpha)$
with $\alpha = \exp(-\kappa(c_i + 1))$.

The wrapping pattern is the **arithmetic heart** of the mass formula — it translates 
the exponential decay of the transient into a modular arithmetic structure on {0,...,p−1}.

In [8]:
# ── S4: Lattice sum decomposition at all 4 physical crossings ──

def lattice_analysis(alpha, p=7):
    """Compute wrapping pattern and lattice sum coefficients."""
    j_arr = np.arange(p)
    Delta = 2 * np.pi * alpha
    
    # Wrapping counts
    n_j = np.round(j_arr * alpha).astype(int)
    
    # Lattice sum coefficients
    S2 = p * (p - 1) * (2*p - 1) // 6  # = 91 for p=7
    M1 = np.sum(j_arr * n_j)
    M0 = np.sum(n_j**2)
    
    # Lattice sum (analytic)
    L_analytic = S2 * Delta**2 - 4*np.pi*Delta*M1 + 4*np.pi**2*M0
    
    # Lattice sum (direct computation)
    wrap_vals = j_arr * Delta
    wrap_vals = np.mod(wrap_vals, 2*np.pi)
    wrap_vals[wrap_vals > np.pi] -= 2*np.pi
    L_direct = np.sum(wrap_vals**2)
    
    # First moment
    mu = np.sum(wrap_vals)
    
    return {
        'alpha': alpha, 'Delta': Delta, 'p': p,
        'n_j': n_j, 'S2': S2, 'M1': M1, 'M0': M0,
        'L_analytic': L_analytic, 'L_direct': L_direct,
        'mu': mu, 'wrap_vals': wrap_vals,
        'total_wraps': n_j[-1],  # max wrapping count
    }

print("="*75)
print("LATTICE SUM ANALYSIS AT ALL PHYSICAL CROSSINGS")
print("="*75)

for name, ci in CROSS.items():
    alpha = np.exp(-KAPPA * (ci + 1))
    res = lattice_analysis(alpha)
    
    print(f"\n{name} (ci={ci}):")
    print(f"  alpha = exp(-kappa*{ci+1}) = {alpha:.6f}")
    print(f"  Delta = 2*pi*alpha = {res['Delta']:.4f}")
    print(f"  7*Delta/(2pi) = {7*alpha:.4f} total wraps")
    print(f"  Wrapping pattern n_j = {list(res['n_j'])}")
    print(f"  S2 = {res['S2']}, M1 = {res['M1']}, M0 = {res['M0']}")
    print(f"  L(analytic)  = {res['L_analytic']:.6f}")
    print(f"  L(direct)    = {res['L_direct']:.6f}")
    print(f"  Match: {abs(res['L_analytic']-res['L_direct']):.2e}")
    print(f"  mu (1st moment) = {res['mu']:.6f}")
    print(f"  wrap values: {[f'{v:.3f}' for v in res['wrap_vals']]}")

# --- Symbolic verification with exact fractions ---
from fractions import Fraction
import sympy as sp

print("\n" + "="*75)
print("EXACT LATTICE SUM FORMULA: L = S2*D^2 - 4*pi*D*M1 + 4*pi^2*M0")
print("="*75)
print(f"{'Channel':>12} {'ci':>4} {'S2':>4} {'M1':>4} {'M0':>4} "
      f"{'n_j pattern':>24}")

for name, ci in CROSS.items():
    alpha = np.exp(-KAPPA * (ci + 1))
    res = lattice_analysis(alpha)
    print(f"  {name:>10} {ci:4d} {res['S2']:4d} {res['M1']:4d} {res['M0']:4d} "
          f"  {list(res['n_j'])}")

# --- Compute 30*L vs actual S_g1_w0 to quantify R_ss effect ---
print("\n" + "="*75)
print("LATTICE SUM vs ACTUAL (quantifying R_ss contribution)")
print("="*75)
for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
    ci_g1 = CROSS[f"{ch}_g1"]
    alpha = np.exp(-KAPPA * (ci_g1 + 1))
    res = lattice_analysis(alpha)
    
    ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
    R_g1_all = R_wrapped[:, ci_g1_idx, lev]
    S_actual = np.sum(R_g1_all**2)
    S_lattice_30 = 30 * res['L_direct']
    
    print(f"\n{ch} g1 (ci={ci_g1}):")
    print(f"  30 * L(Delta) = {S_lattice_30:.2f}")
    print(f"  Actual S_w0   = {S_actual:.2f}")
    print(f"  Ratio actual/lattice = {S_actual/S_lattice_30:.4f}")
    print(f"  Excess from R_ss = {(S_actual-S_lattice_30)/S_actual*100:.1f}%")

LATTICE SUM ANALYSIS AT ALL PHYSICAL CROSSINGS

QUARK_g1 (ci=11):
  alpha = exp(-kappa*12) = 0.436888
  Delta = 2*pi*alpha = 2.7450
  7*Delta/(2pi) = 3.0582 total wraps
  Wrapping pattern n_j = [np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(3)]
  S2 = 91, M1 = 41, M0 = 19
  L(analytic)  = 21.494286
  L(direct)    = 21.494286
  Match: 1.28e-13
  mu (1st moment) = 1.097331
  wrap values: ['0.000', '2.745', '-0.793', '1.952', '-1.586', '1.159', '-2.379']

QUARK_g2 (ci=191):
  alpha = exp(-kappa*192) = 0.000002
  Delta = 2*pi*alpha = 0.0000
  7*Delta/(2pi) = 0.0000 total wraps
  Wrapping pattern n_j = [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
  S2 = 91, M1 = 0, M0 = 0
  L(analytic)  = 0.000000
  L(direct)    = 0.000000
  Match: 0.00e+00
  mu (1st moment) = 0.000232
  wrap values: ['0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000']

LEPTON_g1 (ci=31):
  alpha = exp(-kappa*32) = 0.109897
  

## S5: The Full CP Formula at All 4 Levels

The CP ratio is determined by three ingredients:

1. **Lattice sum** $L(\Delta, p) = S_2\Delta^2 - 4\pi\Delta M_1 + 4\pi^2 M_0$ — purely from wrapping pattern
2. **R_ss correction** — R_ss shifts the lattice, adding ~6% to the g1 sum
3. **Dilution** by $n$ steady-state windows: $\text{CP}^2 = (C_0^2 + r)/(1 + r)$

At each level $k$, the relevant prime is $p_{k+1}$ (the next prime in the covering tower):
- Level 0: $j_1 \in \{0,...,p_1-1\} = \{0,1\}$ — 2 copies
- Level 1: $j_2 \in \{0,...,p_2-1\} = \{0,1,2\}$ — 3 copies
- Level 2: $j_3 \in \{0,...,p_3-1\} = \{0,...,4\}$ — 5 copies
- Level 3: $j_4 \in \{0,...,p_4-1\} = \{0,...,6\}$ — 7 copies

The g1/g2 asymmetry at level $k$ arises from wrapping of $j_{k+1} \cdot \Delta_k$
where $\Delta_k = 2\pi \cdot \exp(-\kappa(c_i+1))$. But the number of copies $p_{k+1}$
and the transient decay rate both change with level!

In [9]:
# ── S5: Per-level lattice analysis ──
# At level k, the decomposition R = R_ss + 2*pi*j_{k+1}*exp(-kappa*(ci+1))
# uses p_{k+1} copies (j from 0 to p_{k+1}-1) and branch count = P4/p_{k+1}.

primes = [2, 3, 5, 7]
P4 = 210

print("="*80)
print("PER-LEVEL LATTICE ANALYSIS")
print("="*80)

for lev_k in range(4):
    p_k = primes[lev_k]         # prime at this level
    n_copies = p_k              # j_{k+1} goes 0..p_k-1
    n_branches_per_j = P4 // p_k  # branches sharing same j_{k+1}
    
    print(f"\n{'─'*80}")
    print(f"Level {lev_k}: prime p_{lev_k+1} = {p_k}, "
          f"copies = {n_copies}, branches/copy = {n_branches_per_j}")
    print(f"{'─'*80}")
    
    # S2 for this level's prime
    S2_lev = n_copies * (n_copies - 1) * (2*n_copies - 1) // 6
    print(f"  S2(p={n_copies}) = {S2_lev}")
    
    for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
        ci_g1 = CROSS[f"{ch}_g1"]
        ci_g2 = CROSS[f"{ch}_g2"]
        alpha_g1 = np.exp(-KAPPA * (ci_g1 + 1))
        alpha_g2 = np.exp(-KAPPA * (ci_g2 + 1))
        Delta_g1 = 2 * np.pi * alpha_g1
        Delta_g2 = 2 * np.pi * alpha_g2
        
        # Wrapping pattern at this level for g1
        j_arr = np.arange(n_copies)
        n_j = np.round(j_arr * alpha_g1).astype(int)
        M1 = int(np.sum(j_arr * n_j))
        M0 = int(np.sum(n_j**2))
        
        # Pure lattice sum
        wrap_g1 = j_arr * Delta_g1
        wrap_g1 = np.mod(wrap_g1, 2*np.pi)
        wrap_g1[wrap_g1 > np.pi] -= 2*np.pi
        L_g1 = np.sum(wrap_g1**2)
        
        # Actual R^2 sums from integration
        ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
        ci_g2_idx = np.where(coprime_cis == ci_g2)[0][0]
        
        R_g1_all = R_wrapped[:, ci_g1_idx, lev_k]  # (210,)
        R_g2_all = R_wrapped[:, ci_g2_idx, lev_k]
        S_g1 = np.sum(R_g1_all**2)
        S_g2 = np.sum(R_g2_all**2)
        
        # Expected pure lattice contribution
        S_lattice = n_branches_per_j * L_g1
        
        # Window-0 CP
        C0 = np.sqrt(S_g1 / S_g2) if S_g2 > 0 else np.inf
        C0_lattice = np.sqrt(S_lattice / S_g2) if S_g2 > 0 else np.inf
        
        # Full CP ratio at this level
        full_rms_g1 = full_rms[0, a3, a7g1, lev_k]
        full_rms_g2 = full_rms[0, a3, a7g2, lev_k]
        CP_full = full_rms_g1 / full_rms_g2 if full_rms_g2 > 0 else np.inf
        
        excess = (S_g1 - S_lattice) / S_g1 * 100 if S_g1 > 0 else 0
        
        print(f"\n  {ch} at level {lev_k}:")
        print(f"    g1 (ci={ci_g1}): alpha={alpha_g1:.6f}, "
              f"Delta={Delta_g1:.4f}, wraps={n_copies*alpha_g1:.3f}")
        print(f"    n_j = {list(n_j)}, M1={M1}, M0={M0}")
        print(f"    L(pure lattice) = {L_g1:.4f}")
        print(f"    S_lattice ({n_branches_per_j}*L) = {S_lattice:.2f}")
        print(f"    S_actual                 = {S_g1:.2f}")
        print(f"    R_ss excess = {excess:.1f}%")
        print(f"    C0(actual) = {C0:.4f}, C0(lattice) = {C0_lattice:.4f}")
        print(f"    CP_full(level {lev_k}) = {CP_full:.6f}")

PER-LEVEL LATTICE ANALYSIS

────────────────────────────────────────────────────────────────────────────────
Level 0: prime p_1 = 2, copies = 2, branches/copy = 105
────────────────────────────────────────────────────────────────────────────────
  S2(p=2) = 1

  QUARK at level 0:
    g1 (ci=11): alpha=0.436888, Delta=2.7450, wraps=0.874
    n_j = [np.int64(0), np.int64(0)], M1=0, M0=0
    L(pure lattice) = 7.5353
    S_lattice (105*L) = 791.21
    S_actual                 = 787.65
    R_ss excess = -0.5%
    C0(actual) = 176.4485, C0(lattice) = 176.8465
    CP_full(level 0) = 36.013297

  LEPTON at level 0:
    g1 (ci=31): alpha=0.109897, Delta=0.6905, wraps=0.220
    n_j = [np.int64(0), np.int64(0)], M1=0, M0=0
    L(pure lattice) = 0.4768
    S_lattice (105*L) = 50.06
    S_actual                 = 48.67
    R_ss excess = -2.9%
    C0(actual) = 8.8358, C0(lattice) = 8.9617
    CP_full(level 0) = 6.390830

───────────────────────────────────────────────────────────────────────────────

In [10]:
# ── S5b: Compact summary table ──
print("COMPACT SUMMARY: Lattice coefficients and CP ratios at all 4 levels")
print("="*90)
print(f"{'Lev':>3} {'Ch':>6} {'ci_g1':>5} {'p':>2} {'alpha':>8} "
      f"{'M1':>4} {'M0':>4} {'L':>8} {'excess%':>8} {'C0':>8} {'CP_full':>8}")
print("-"*90)

for lev_k in range(4):
    p_k = primes[lev_k]
    n_br_per_j = P4 // p_k
    
    for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
        ci_g1 = CROSS[f"{ch}_g1"]
        ci_g2 = CROSS[f"{ch}_g2"]
        alpha = np.exp(-KAPPA * (ci_g1 + 1))
        Delta = 2 * np.pi * alpha
        
        j_arr = np.arange(p_k)
        n_j = np.round(j_arr * alpha).astype(int)
        M1 = int(np.sum(j_arr * n_j))
        M0 = int(np.sum(n_j**2))
        
        wrap_v = j_arr * Delta
        wrap_v = np.mod(wrap_v, 2*np.pi)
        wrap_v[wrap_v > np.pi] -= 2*np.pi
        L = np.sum(wrap_v**2)
        
        ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
        ci_g2_idx = np.where(coprime_cis == ci_g2)[0][0]
        S_g1 = np.sum(R_wrapped[:, ci_g1_idx, lev_k]**2)
        S_g2 = np.sum(R_wrapped[:, ci_g2_idx, lev_k]**2)
        S_lat = n_br_per_j * L
        
        excess = (S_g1 - S_lat) / S_g1 * 100 if S_g1 > 0 else 0
        C0 = np.sqrt(S_g1/S_g2) if S_g2 > 0 else np.inf
        
        fr_g1 = full_rms[0, a3, a7g1, lev_k]
        fr_g2 = full_rms[0, a3, a7g2, lev_k]
        CP = fr_g1/fr_g2 if fr_g2 > 0 else np.inf
        
        print(f"{lev_k:3d} {ch:>6} {ci_g1:5d} {p_k:2d} {alpha:8.5f} "
              f"{M1:4d} {M0:4d} {L:8.3f} {excess:8.1f} {C0:8.4f} {CP:8.6f}")

COMPACT SUMMARY: Lattice coefficients and CP ratios at all 4 levels
Lev     Ch ci_g1  p    alpha   M1   M0        L  excess%       C0  CP_full
------------------------------------------------------------------------------------------
  0  QUARK    11  2  0.43689    0    0    7.535     -0.5 176.4485 36.013297
  0 LEPTON    31  2  0.10990    0    0    0.477     -2.9   8.8358 6.390830
  1  QUARK    11  3  0.43689    2    1    8.164     -6.5  97.3487 19.840202
  1 LEPTON    31  3  0.10990    0    0    2.384     49.6   6.2340 5.815592
  2  QUARK    11  5  0.43689   13    6   14.490      4.3  29.9278 6.171884
  2 LEPTON    31  5  0.10990    0    0   14.304     35.1   4.6889 4.264373
  3  QUARK    11  7  0.43689   41   19   21.494      6.3   5.8124 1.538827
  3 LEPTON    31  7  0.10990   11    2   26.897      6.0   6.1771 1.942172


In [11]:
# ── S5c: Dilution formula verification at all 4 levels  ──
# CP^2 = (C0^2 + r) / (1 + r) where r = S_wN / S_g2_w0

print("DILUTION FORMULA VERIFICATION AT ALL 4 LEVELS")
print("="*90)
print(f"{'Lev':>3} {'Ch':>6} {'C0':>8} {'r':>10} {'CP_pred':>10} "
      f"{'CP_act':>10} {'err%':>8} {'n_wN_g1':>8} {'n_wN_g2':>8}")
print("-"*90)

for lev_k in range(4):
    for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
        ci_g1 = CROSS[f"{ch}_g1"]
        ci_g2 = CROSS[f"{ch}_g2"]
        
        ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
        ci_g2_idx = np.where(coprime_cis == ci_g2)[0][0]
        
        # Window-0 sums
        S_g1_w0 = np.sum(R_wrapped[:, ci_g1_idx, lev_k]**2)
        S_g2_w0 = np.sum(R_wrapped[:, ci_g2_idx, lev_k]**2)
        C0 = np.sqrt(S_g1_w0 / S_g2_w0) if S_g2_w0 > 0 else np.inf
        
        # Window >= 1 sums (average g1 and g2)
        wN_g1 = ((coprime_cis >= 210) &
                 (ci_a3 == a3) & (ci_a5 == 0) & (ci_a7 == a7g1))
        wN_g2 = ((coprime_cis >= 210) &
                 (ci_a3 == a3) & (ci_a5 == 0) & (ci_a7 == a7g2))
        
        n_g1 = np.sum(wN_g1)
        n_g2 = np.sum(wN_g2)
        
        S_wN_g2 = np.sum(R_wrapped[:, wN_g2, lev_k]**2)
        S_wN_g1 = np.sum(R_wrapped[:, wN_g1, lev_k]**2)
        S_wN = (S_wN_g1 + S_wN_g2) / 2  # average since CP_wN=1
        
        r = S_wN / S_g2_w0 if S_g2_w0 > 0 else np.inf
        C0_sq = C0**2
        
        CP_pred = np.sqrt((C0_sq + r) / (1 + r))
        
        # Actual full CP
        fr_g1 = full_rms[0, a3, a7g1, lev_k]
        fr_g2 = full_rms[0, a3, a7g2, lev_k]
        CP_act = fr_g1 / fr_g2 if fr_g2 > 0 else np.inf
        
        err = abs(CP_pred - CP_act) / CP_act * 100
        
        print(f"{lev_k:3d} {ch:>6} {C0:8.4f} {r:10.4f} {CP_pred:10.6f} "
              f"{CP_act:10.6f} {err:8.4f} {n_g1:8d} {n_g2:8d}")

DILUTION FORMULA VERIFICATION AT ALL 4 LEVELS
Lev     Ch       C0          r    CP_pred     CP_act     err%  n_wN_g1  n_wN_g2
------------------------------------------------------------------------------------------
  0  QUARK 176.4485    22.5227  36.394141  36.013297   1.0575       23       22
  0 LEPTON   8.8358     0.9344   6.390831   6.390830   0.0000       23       23
  1  QUARK  97.3487    22.6308  20.049748  19.840202   1.0562       23       22
  1 LEPTON   6.2340     0.1536   5.815594   5.815592   0.0000       23       23
  2  QUARK  29.9278    22.6175   6.235531   6.171884   1.0312       23       22
  2 LEPTON   4.6889     0.2212   4.264371   4.264373   0.0001       23       23
  3  QUARK   5.8124    22.4664   1.548251   1.538827   0.6124       23       22
  3 LEPTON   6.1771    12.4041   1.942174   1.942172   0.0001       23       23


## S6: The R_ss Correction — What the Lattice Doesn't Capture

The pure lattice sum $L(\Delta, p)$ accounts for ~94% of the window-0 g1 signal at level 3.
The remaining ~6% comes from R_ss shifting the wrapped pattern. This correction involves:

1. **Quadratic term**: $7 \cdot \sum R_{ss}^2$ (always positive)
2. **Cross term**: $2\mu \cdot \sum R_{ss}$ (sign depends on lattice first moment)
3. **Boundary effects**: R_ss shifts some points past wrapping boundaries (typically negative)

These largely cancel, leaving a ~6% net excess. Can we characterize this correction analytically?

In [12]:
# ── S6: R_ss correction analysis ──
# For each channel at level 3: decompose the excess into components.

print("R_SS CORRECTION ANATOMY (Level 3)")
print("="*75)

for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
    ci_g1 = CROSS[f"{ch}_g1"]
    ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
    
    alpha = np.exp(-KAPPA * (ci_g1 + 1))
    Delta = 2 * np.pi * alpha
    
    # R_ss distribution (j4=0 branches)
    j4_0 = branches_arr[:, 3] == 0
    R_ss = R_stack[j4_0, ci_g1_idx, lev]  # 30 values
    n_br_30 = len(R_ss)
    
    # Pure lattice values (R_ss = 0)
    j_arr = np.arange(7)
    lat_raw = j_arr * Delta
    lat_wrap = np.mod(lat_raw, 2*np.pi)
    lat_wrap[lat_wrap > np.pi] -= 2*np.pi
    L = np.sum(lat_wrap**2)
    mu = np.sum(lat_wrap)  # first moment
    
    # Actual per-branch sums
    S_per_branch = np.zeros(n_br_30)
    for i, b in enumerate(R_ss):
        vals = b + j_arr * Delta
        w = np.mod(vals, 2*np.pi)
        w[w > np.pi] -= 2*np.pi
        S_per_branch[i] = np.sum(w**2)
    
    S_actual = np.sum(S_per_branch)
    S_lattice = n_br_30 * L
    excess = S_actual - S_lattice
    
    # Perturbative prediction (no boundary effects)
    perturbative = 2 * mu * np.sum(R_ss) + 7 * np.sum(R_ss**2)
    boundary_correction = excess - perturbative
    
    print(f"\n{ch} g1 (ci={ci_g1}, alpha={alpha:.4f}):")
    print(f"  R_ss stats: mean={np.mean(R_ss):.4f}, std={np.std(R_ss):.4f}")
    print(f"  <R_ss^2> = {np.mean(R_ss**2):.4f}")
    print(f"  Lattice L = {L:.4f},  mu = {mu:.4f}")
    print(f"  30*L = {S_lattice:.2f}")
    print(f"  Actual = {S_actual:.2f}")
    print(f"  Excess = {excess:.2f}  ({excess/S_actual*100:.1f}%)")
    print(f"    Perturbative pred (quad+cross) = {perturbative:.2f}")
    print(f"      Quadratic: 7*sum(R_ss^2) = {7*np.sum(R_ss**2):.2f}")
    print(f"      Cross:     2*mu*sum(R_ss) = {2*mu*np.sum(R_ss):.2f}")
    print(f"    Boundary correction          = {boundary_correction:.2f}")
    print(f"    Boundary as % of perturbative = "
          f"{abs(boundary_correction)/abs(perturbative)*100:.0f}%")
    
    # How uniform is the per-branch contribution?
    print(f"  Per-branch sum: mean={np.mean(S_per_branch):.3f}, "
          f"std={np.std(S_per_branch):.3f}, "
          f"cv={np.std(S_per_branch)/np.mean(S_per_branch):.3f}")

# --- The crucial question: ratio S_actual/(30*L) for both channels ---
print(f"\n{'='*75}")
print("KEY RATIO: S_actual / (30*L) at level 3")
ratios = {}
for ch in ['QUARK', 'LEPTON']:
    ci_g1 = CROSS[f"{ch}_g1"]
    ci_g1_idx = np.where(coprime_cis == ci_g1)[0][0]
    alpha = np.exp(-KAPPA * (ci_g1 + 1))
    Delta = 2 * np.pi * alpha
    j_arr = np.arange(7)
    lat = j_arr * Delta
    lat = np.mod(lat, 2*np.pi)
    lat[lat > np.pi] -= 2*np.pi
    L = np.sum(lat**2)
    S = np.sum(R_wrapped[:, ci_g1_idx, lev]**2)
    ratio = S / (30 * L)
    ratios[ch] = ratio
    print(f"  {ch}: {ratio:.6f}")
print(f"  Ratio of ratios: {ratios['QUARK']/ratios['LEPTON']:.6f}")

R_SS CORRECTION ANATOMY (Level 3)

QUARK g1 (ci=11, alpha=0.4369):
  R_ss stats: mean=0.8598, std=0.3256
  <R_ss^2> = 0.8453
  Lattice L = 21.4943,  mu = 1.0973
  30*L = 644.83
  Actual = 688.19
  Excess = 43.37  (6.3%)
    Perturbative pred (quad+cross) = 234.12
      Quadratic: 7*sum(R_ss^2) = 177.51
      Cross:     2*mu*sum(R_ss) = 56.61
    Boundary correction          = -190.75
    Boundary as % of perturbative = 81%
  Per-branch sum: mean=22.940, std=0.268, cv=0.012

LEPTON g1 (ci=31, alpha=0.1099):
  R_ss stats: mean=0.5716, std=0.4897
  <R_ss^2> = 0.5665
  Lattice L = 26.8968,  mu = 1.9342
  30*L = 806.90
  Actual = 858.53
  Excess = 51.63  (6.0%)
    Perturbative pred (quad+cross) = 185.30
      Quadratic: 7*sum(R_ss^2) = 118.96
      Cross:     2*mu*sum(R_ss) = 66.34
    Boundary correction          = -133.67
    Boundary as % of perturbative = 72%
  Per-branch sum: mean=28.618, std=1.093, cv=0.038

KEY RATIO: S_actual / (30*L) at level 3
  QUARK: 1.067251
  LEPTON: 1.063984

## S7: What Blocks a Closed-Form CP?

The analytic CP formula requires two ingredients:

1. **$C_0$** — the window-0 CP ratio. This is $\sqrt{S_{g1}^{w0} / S_{g2}^{w0}}$ where:
   - $S_{g1}^{w0}$ = lattice sum $\times$ 30 $\times$ (1 + $\eta$), with $\eta \approx 0.065$ (R_ss correction)
   - $S_{g2}^{w0}$ = 210 $\times$ $\langle R_{ss}^2 \rangle_{g2}$ (steady-state at g2 crossing)

2. **$r$** — the dilution ratio. This is $n \times \sigma_{ss}^2 / \sigma_{w0}^2$ where:
   - $n$ = number of post-window-0 crossings
   - $\sigma_{ss}^2$ = per-crossing steady-state R²
   - $\sigma_{w0}^2$ = window-0 g2 R² per crossing

**What is analytically expressible**:
- The lattice sum $L(\Delta, p) = S_2\Delta^2 - 4\pi\Delta M_1 + 4\pi^2 M_0$ — fully determined by $\{2,3,5,7\}$ via $\Delta = 2\pi e^{-\kappa(c_i+1)}$
- The wrapping coefficients $(M_1, M_0)$ — integers from $n_j = \text{round}(j \cdot e^{-\kappa(c_i+1)})$
- The CRT crossing positions $c_i \in \{11, 31, 61, 191\}$ — from the Chinese Remainder Theorem on 210

**What requires cascade dynamics**:
- $R_{ss}$ at each crossing — the steady-state of the nonlinear cascade ODE
- The R_ss correction $\eta$ — a delicate balance of quadratic, cross, and boundary terms
- The ratio $\sigma_{ss}^2 / \sigma_{w0}^2$ — depends on transient decay at the g2 crossing

**The structural result**: Fermion masses arise from a *wrapped lattice sum* at CRT-determined 
crossing positions, *diluted* by steady-state windows. The lattice is fully determined 
by number theory; the dilution involves cascade dynamics that resist closed-form solution.

This is NOT a failure — it identifies exactly WHERE the cascade dynamics are needed and 
HOW MUCH of the mass formula is purely arithmetic. At level 3, **94% of the g1 signal 
is captured by the lattice alone**.

In [13]:
# ── S7: The complete structural formula ──
# Assemble all ingredients and test the "lattice-only" C0 prediction.

print("THE STRUCTURAL MASS FORMULA")
print("="*75)
print()
print("CP^2 = (C0^2 + r) / (1 + r)")
print()
print("where C0 = sqrt(S_g1_w0 / S_g2_w0)  [window-0 ratio]")
print("      r = S_wN / S_g2_w0              [dilution parameter]")
print()

# --- What if we use the lattice C0 instead of actual C0? ---
print("LATTICE-ONLY MASS PREDICTIONS (no R_ss, level 3)")
print("-"*75)

from solenoid_algebra import SA, X4, X3, X2, LAM7, X4_LEP, SM_TARGETS

for ch, (a3, a7g1, a7g2) in CP_PAIRS.items():
    ci_g1 = CROSS[f"{ch}_g1"]
    ci_g2 = CROSS[f"{ch}_g2"]
    
    # Lattice-based C0
    alpha = np.exp(-KAPPA * (ci_g1 + 1))
    Delta = 2 * np.pi * alpha
    j_arr = np.arange(7)
    lat = j_arr * Delta
    lat = np.mod(lat, 2*np.pi)
    lat[lat > np.pi] -= 2*np.pi
    L = np.sum(lat**2)
    
    ci_g2_idx = np.where(coprime_cis == ci_g2)[0][0]
    S_g2_w0 = np.sum(R_wrapped[:, ci_g2_idx, lev]**2)
    
    C0_lattice = np.sqrt(30 * L / S_g2_w0)
    C0_actual = cp_w0[ch]
    
    # r from data
    wN_g2_mask = ((coprime_cis >= 210) & 
                  (ci_a3 == a3) & (ci_a5 == 0) & (ci_a7 == a7g2))
    S_wN = np.sum(R_wrapped[:, wN_g2_mask, lev]**2)
    r = S_wN / S_g2_w0
    
    CP_lattice = np.sqrt((C0_lattice**2 + r) / (1 + r))
    CP_actual = cp_full[ch]
    
    print(f"\n{ch}:")
    print(f"  C0_lattice = {C0_lattice:.4f}  (C0_actual = {C0_actual:.4f}, "
          f"diff = {(C0_lattice/C0_actual - 1)*100:+.1f}%)")
    print(f"  CP_lattice = {CP_lattice:.6f}  (CP_actual = {CP_actual:.6f}, "
          f"diff = {(CP_lattice/CP_actual - 1)*100:+.2f}%)")

# --- Full mass predictions: lattice vs actual ---
print("\n" + "="*75)
print("MASS RATIO PREDICTIONS: LATTICE vs ACTUAL vs PDG")
print("="*75)

# Level 2 and 3 CP ratios (actual)
R4_q = cp_full['QUARK']   # level 3
R3_q_rms = full_rms[0, 1, CP_PAIRS['QUARK'][1], 2] / \
           full_rms[0, 1, CP_PAIRS['QUARK'][2], 2]  # level 2
R4_l = cp_full['LEPTON']  # level 3
R3_l_rms = full_rms[0, 0, CP_PAIRS['LEPTON'][1], 2] / \
           full_rms[0, 0, CP_PAIRS['LEPTON'][2], 2]  # level 2

# Mass predictions
ms_md_pred = R4_q**X4
mmu_me_pred = R4_l**X4_LEP

# PDG targets
ms_md_pdg = SM_TARGETS.get('ms_md', (20.0, 1.0))
mmu_me_pdg = SM_TARGETS.get('mmu_me', (206.768, 0.0))

print(f"\n  m_s/m_d:  predicted = {ms_md_pred:.2f}, "
      f"PDG = {ms_md_pdg[0]:.1f} +/- {ms_md_pdg[1]}")
print(f"           error = {(ms_md_pred/ms_md_pdg[0] - 1)*100:+.1f}%")
print(f"           (CP = {R4_q:.6f}, X4 = {X4:.4f})")

print(f"\n  m_mu/m_e: predicted = {mmu_me_pred:.2f}, "
      f"PDG = {mmu_me_pdg[0]:.3f}")
print(f"           error = {(mmu_me_pred/mmu_me_pdg[0] - 1)*100:+.1f}%")
print(f"           (CP = {R4_l:.6f}, X4_lep = {X4_LEP:.4f})")

print(f"\n  NOTE: These use T=5001 integration. Mass predictions improve")
print(f"  with T as the CP ratio slowly converges. The structural formula")
print(f"  shows WHERE the convergence comes from: each additional window")
print(f"  adds one unit of dilution to r, bringing CP closer to its")
print(f"  asymptotic value.")

THE STRUCTURAL MASS FORMULA

CP^2 = (C0^2 + r) / (1 + r)

where C0 = sqrt(S_g1_w0 / S_g2_w0)  [window-0 ratio]
      r = S_wN / S_g2_w0              [dilution parameter]

LATTICE-ONLY MASS PREDICTIONS (no R_ss, level 3)
---------------------------------------------------------------------------

QUARK:
  C0_lattice = 5.6263  (C0_actual = 5.8124, diff = -3.2%)
  CP_lattice = 1.527994  (CP_actual = 1.538827, diff = -0.70%)

LEPTON:
  C0_lattice = 5.9885  (C0_actual = 6.1771, diff = -3.1%)
  CP_lattice = 1.897590  (CP_actual = 1.942172, diff = -2.30%)

MASS RATIO PREDICTIONS: LATTICE vs ACTUAL vs PDG

  m_s/m_d:  predicted = 26.92, PDG = 20.0 +/- 1.0
           error = +34.6%
           (CP = 1.538827, X4 = 7.6394)

  m_mu/m_e: predicted = 177.11, PDG = 206.768
           error = -14.3%
           (CP = 1.942172, X4_lep = 7.7986)

  NOTE: These use T=5001 integration. Mass predictions improve
  with T as the CP ratio slowly converges. The structural formula
  shows WHERE the convergence c

In [14]:
# ── S7b: Verify wrapping pattern robustness ──
# The wrapping pattern n_j = round(j*alpha) is determined by the primes.
# Check: how far are the j*alpha values from the nearest half-integer?

print("WRAPPING PATTERN ROBUSTNESS")
print("="*70)
print("The wrapping pattern is robust if j*alpha is far from half-integers.")
print()

for name, ci in CROSS.items():
    alpha = np.exp(-KAPPA * (ci + 1))
    j_arr = np.arange(7)
    ja = j_arr * alpha
    n_j = np.round(ja).astype(int)
    
    # Distance to nearest half-integer: d = |j*alpha - n_j| - 0 is AT boundary
    distances = np.abs(ja - n_j)  # distance from nearest integer
    # Half-integer boundary: distance = 0.5 - distances
    margins = 0.5 - distances  # how far from the wrapping boundary
    
    min_margin = np.min(margins[1:])  # exclude j=0
    critical_j = np.argmin(margins[1:]) + 1
    
    print(f"{name} (ci={ci}):  alpha = {alpha:.6f}")
    print(f"  n_j = {list(n_j)}")
    print(f"  j*alpha: {[f'{v:.4f}' for v in ja]}")
    print(f"  Margins: {[f'{v:.4f}' for v in margins]}")
    print(f"  Min margin = {min_margin:.4f} at j={critical_j}")
    print()

# ── Compact identity table ──
print("="*70)
print("WRAPPING COEFFICIENT TABLE (level 3, p=7)")
print("="*70)
print(f"{'Crossing':>12} {'ci':>4} {'n_j':>24} {'M1':>4} {'M0':>4} "
      f"{'L':>8} {'frac':>6}")
print("-"*70)

for name, ci in CROSS.items():
    alpha = np.exp(-KAPPA * (ci + 1))
    j_arr = np.arange(7)
    n_j = np.round(j_arr * alpha).astype(int)
    M1 = int(np.sum(j_arr * n_j))
    M0 = int(np.sum(n_j**2))
    Delta = 2*np.pi*alpha
    L = 91*Delta**2 - 4*np.pi*Delta*M1 + 4*np.pi**2*M0
    
    # Fraction of S_g1_w0 captured by lattice
    ci_idx = np.where(coprime_cis == ci)[0][0]
    S_actual = np.sum(R_wrapped[:, ci_idx, 3]**2)
    frac = 30*L / S_actual if S_actual > 0 else 0
    
    n_str = str(list(n_j)).replace('np.int64(', '').replace(')', '')
    print(f"{name:>12} {ci:4d} {n_str:>24} {M1:4d} {M0:4d} "
          f"{L:8.3f} {frac:6.3f}")

WRAPPING PATTERN ROBUSTNESS
The wrapping pattern is robust if j*alpha is far from half-integers.

QUARK_g1 (ci=11):  alpha = 0.436888
  n_j = [np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(3)]
  j*alpha: ['0.0000', '0.4369', '0.8738', '1.3107', '1.7476', '2.1844', '2.6213']
  Margins: ['0.5000', '0.0631', '0.3738', '0.1893', '0.2476', '0.3156', '0.1213']
  Min margin = 0.0631 at j=1

QUARK_g2 (ci=191):  alpha = 0.000002
  n_j = [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
  j*alpha: ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000', '0.0000', '0.0000']
  Margins: ['0.5000', '0.5000', '0.5000', '0.5000', '0.5000', '0.5000', '0.5000']
  Min margin = 0.5000 at j=6

LEPTON_g1 (ci=31):  alpha = 0.109897
  n_j = [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1)]
  j*alpha: ['0.0000', '0.1099', '0.2198', '0.3297', '0.4396', '0.5495', '0.6594']
  Margins: ['0.5000', 

In [15]:
# ── Scorecard ──
print("NB106 SCORECARD")
print("=" * 65)
print()
print("STRUCTURAL RESULTS (no new identity numbers)")
print("-" * 65)
print()
print("1. DILUTION FORMULA DERIVATION")
print("   CP^2 = (C0^2 + r) / (1 + r)")
print("   Derived from window-0 concentration principle.")
print("   EXACT at all 4 levels (< 0.001% error when")
print("   window counts match; ~1% for quark due to")
print("   finite-T window asymmetry 23 vs 22).")
print("   This PROVES the NB97 empirical formula (#217)")
print("   from CRT window structure.")
print()
print("2. LATTICE SUM FORMULA")
print("   L(D,p) = S2*D^2 - 4*pi*D*M1 + 4*pi^2*M0")
print("   S2 = p(p-1)(2p-1)/6 = 91 for p=7")
print("   (M1, M0) integers from wrapping pattern")
print("     n_j = round(j * exp(-kappa*(ci+1)))")
print()
print("   Level 3 wrapping coefficients:")
print("   QUARK g1 (ci=11):  n={0,0,1,1,2,2,3}  M1=41 M0=19")
print("   LEPTON g1 (ci=31): n={0,0,0,0,0,1,1}  M1=11 M0=2")
print("   Both g2:           n={0,...,0}          M1=0  M0=0")
print("   Pure lattice captures 94% of window-0 g1 signal.")
print()
print("3. R_ss CORRECTION")
print("   S_actual / (30*L) = 1.067 (Q), 1.064 (L)")
print("   Nearly channel-independent (~0.3% match).")
print("   Correction anatomy: large perturbative terms")
print("   (quad + cross ~ 30%) are 72-81% cancelled by")
print("   wrapping boundary effects, leaving ~6.5% net.")
print()
print("4. PER-BRANCH HOMOGENEITY")
print("   QUARK: cv = 1.2%  (wrapping strongly homogenizes)")
print("   LEPTON: cv = 3.8%")
print()
print("5. CLOSED-FORM BLOCKED BY CASCADE DYNAMICS")
print("   Lattice sum L(D): fully from {2,3,5,7}.")
print("   R_ss correction: requires cascade ODE solution.")
print("   Dilution parameter r: requires steady-state R_ss.")
print("   A fully closed-form mass formula would need")
print("   an analytic solution to the nonlinear cascade.")
print()
print("=" * 65)
print("NEW IDENTITIES: 0 (honest NULL — structural insight)")
print("Running total: 228 predictions/identities, 0 free parameters")
print("=" * 65)

NB106 SCORECARD

STRUCTURAL RESULTS (no new identity numbers)
-----------------------------------------------------------------

1. DILUTION FORMULA DERIVATION
   CP^2 = (C0^2 + r) / (1 + r)
   Derived from window-0 concentration principle.
   EXACT at all 4 levels (< 0.001% error when
   window counts match; ~1% for quark due to
   finite-T window asymmetry 23 vs 22).
   This PROVES the NB97 empirical formula (#217)
   from CRT window structure.

2. LATTICE SUM FORMULA
   L(D,p) = S2*D^2 - 4*pi*D*M1 + 4*pi^2*M0
   S2 = p(p-1)(2p-1)/6 = 91 for p=7
   (M1, M0) integers from wrapping pattern
     n_j = round(j * exp(-kappa*(ci+1)))

   Level 3 wrapping coefficients:
   QUARK g1 (ci=11):  n={0,0,1,1,2,2,3}  M1=41 M0=19
   LEPTON g1 (ci=31): n={0,0,0,0,0,1,1}  M1=11 M0=2
   Both g2:           n={0,...,0}          M1=0  M0=0
   Pure lattice captures 94% of window-0 g1 signal.

3. R_ss CORRECTION
   S_actual / (30*L) = 1.067 (Q), 1.064 (L)
   Nearly channel-independent (~0.3% match).
   Corr